# One-session data import template

The figure workflow uses explicit function calls: `build_config(...)` declares
inputs, `preview_session(config)` checks the selection, and
`load_figure_data(config, ...)` returns a `FigureData` object. Loading and plotting
are separate, so plot parameters can change without reading the recording again.

`FigureData` contains the session, raw and processed pose, a selected animal's
`position(time, space, keypoint)`, one tracked `point(time, space)`, and sorted
trial/event tables. Spatial units are **pixels**, and time stays on the
**acquisition clock in seconds**. Edit the paths below before running this demo.


In [ ]:
#=== 1| Find the checkout from Jupyter's working directory ========
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
source_root = next(
    (candidate for parent in candidates for candidate in (parent, parent / "src")
     if (candidate / "movement_figures").is_dir()),
    None,
)
if source_root is None:
    raise RuntimeError("Open this notebook from the data-conduit checkout.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

#=== 2| Import functions with explicit configuration and result types ========
import numpy as np
import pandas as pd
from IPython.display import display
from movement_figures.data_template.loading import (
    FigureData, SessionData, build_config, preview_session, load_figure_data,
)


## 1. Declare root, scope and contents

The default hierarchy is `ROOT / mouseID / day / SESSION`. If the root directly contains session folders, set `LEVEL_NAMES = ()`. `SESSION` is the exact final folder name. Set `LEVEL_SELECTORS`, for example `{"l0_selector": "mouse_name"}`, to narrow intermediate folders when needed.

The figure notebooks require `trials` and `dlc`. Keep `events` to inspect raw event records; other Q_C sources can be added. VideoData is an internal DLC timing dependency, so it need not be explicitly included. This cell constructs a configuration and performs no filesystem reads.

In [ ]:
#=== 1| Declare the session inputs ========
ROOT = Path("/path/to/BonsaiOutput/Training")
SESSION = "REPLACE_WITH_SESSION_FOLDER"  # Folder name only, not a full path.
LEVEL_NAMES = ("mouseID", "day")
LEVEL_SELECTORS = None
SOURCES = ("trials", "events", "dlc")
INDIVIDUAL = "individual_0"
TRACKING_KEYPOINT = "body"

#=== 2| Build a configuration without opening any source files ========
config = build_config(
    root=ROOT,
    session=SESSION,
    level_names=LEVEL_NAMES,
    level_selectors=LEVEL_SELECTORS,
    sources=SOURCES,
)
display(config)


## 2. Preview selection, then load

Preview reads directory names, without invoking source readers. It must find **exactly one session**; ambiguous matches require narrower selectors. The loader repeats this check before loading and checks coverage for trials, position and confidence.

In [ ]:
#=== 1| Preview the exact session before reading its source data ========
preview = preview_session(config=config)
display(preview)


In [ ]:
def session_tables(session_data: SessionData, *, n_rows: int = 5) -> dict[str, pd.DataFrame]:
    """Return small tables for inspection without modifying the loaded session.

    Parameters
    ----------
    session_data : SessionData
        The session returned by the loader, including coverage and source tables.
    n_rows : int
        Maximum trial/event rows to include in the preview.

    Returns
    -------
    dict[str, pandas.DataFrame]
        Copies of the session manifest, stream coverage, trial preview and,
        when requested during loading, raw event preview.
    """
    #=== 1| Validate the preview size ========
    if isinstance(n_rows, bool) or not isinstance(n_rows, int) or n_rows < 1:
        raise ValueError("n_rows must be a positive integer.")

    #=== 2| Collect table copies as an explicit return value ========
    tables = {
        "Session": session_data.loaded.session_manifest.copy(),
        "Stream coverage": session_data.loaded.stream_coverage.copy(),
        "Trials": session_data.trials.head(n_rows).copy(),
    }
    if session_data.events is not None:
        tables["Raw events"] = session_data.events.head(n_rows).copy()
    return tables


#=== 3| Load and process once using named input arguments ========
figure_data = load_figure_data(
    config=config,
    individual=INDIVIDUAL,
    tracking_keypoint=TRACKING_KEYPOINT,
    confidence_threshold=0.9,
    max_gap_frames=None,
    smoothing_window=None,
)

#=== 4| Display the returned tables separately from loading ========
inspection_tables = session_tables(session_data=figure_data.session_data, n_rows=5)
for label, table in inspection_tables.items():
    print(label)
    display(table)


In [ ]:
def pose_summary(pose) -> dict:
    """Return coordinate names, clock bounds and missing-value counts for a pose.

    Parameters
    ----------
    pose : xarray.Dataset
        A movement-schema dataset with position(time, space, keypoint, individual).

    Returns
    -------
    dict
        Names, dimensions, pixel/second units, clock bounds and missing coordinate
        count. Missing coordinates count every scalar x/y value, not frames.
    """
    #=== 1| Read the schema and clock without changing the data ========
    return {
        "keypoints": pose.keypoint.values.tolist(),
        "individuals": pose.individual.values.tolist(),
        "position_dimensions": tuple(pose.position.dims),
        "position_shape": tuple(pose.position.shape),
        "spatial_unit": pose.attrs.get("spatial_unit", "pixels"),
        "time_unit": pose.attrs.get("time_unit", "seconds"),
        "clock_range_s": (float(pose.time[0]), float(pose.time[-1])),
        "missing_coordinates": int(pose.position.isnull().sum()),
    }


#=== 2| Inspect the explicitly returned raw and processed pose ========
raw_summary = pose_summary(pose=figure_data.raw_pose)
processed_summary = pose_summary(pose=figure_data.pose)
display(pd.DataFrame({"Raw": raw_summary, "Processed": processed_summary}))


## 3. Compare optional movement processing

The call to `load_figure_data` exposes `confidence_threshold`, `max_gap_frames`
and `smoothing_window` as keyword inputs. Set them there and rerun loading when
processing changes. The unprocessed `raw_pose` remains available for comparison.

A gap limit counts consecutive missing **samples**, not elapsed seconds.
Smoothing uses an odd rolling-median window of at least three samples.
Interpolation and smoothing are disabled in this example. The comparison
function below checks the two returned arrays without changing either one.


In [ ]:
def compare_pose_processing(raw_pose, processed_pose) -> pd.DataFrame:
    """Compare raw and processed missing-coordinate counts on the same schema.

    Parameters
    ----------
    raw_pose, processed_pose : xarray.Dataset
        Unprocessed and processed datasets from the same loaded recording.

    Returns
    -------
    pandas.DataFrame
        One row for each input, reporting total, missing and available scalar
        coordinates. This is an inspection table, not a tracking-quality score.
    """
    #=== 1| Confirm that the comparison covers the same coordinate grid ========
    if raw_pose.position.dims != processed_pose.position.dims:
        raise ValueError("The raw and processed position dimensions must match.")
    for dimension in raw_pose.position.dims:
        if not raw_pose[dimension].equals(processed_pose[dimension]):
            raise ValueError(f"The coordinate grid differs on {dimension!r}.")

    #=== 2| Return counts without treating individual coordinates as frames ========
    records = []
    for label, pose in (("Raw", raw_pose), ("Processed", processed_pose)):
        missing = int(pose.position.isnull().sum())
        records.append({
            "pose": label,
            "total_coordinates": pose.position.size,
            "missing_coordinates": missing,
            "available_coordinates": pose.position.size - missing,
        })
    return pd.DataFrame.from_records(records).set_index("pose")


#=== 3| Inspect the returned comparison and movement's processing log ========
processing_comparison = compare_pose_processing(
    raw_pose=figure_data.raw_pose,
    processed_pose=figure_data.pose,
)
display(processing_comparison)
display(figure_data.pose.position.attrs)


Pass `figure_data.position`, `figure_data.point`, `figure_data.trials` and
`figure_data.events` explicitly to the plotting/measurement functions in the
figure notebooks. Those functions return arrays, tables or `(figure, axes)`;
example calls display the returned figures. See
[the function map](../MOVEMENT_FUNCTIONS.md) for movement operations and the
additional analysis/plotting code provided by this project.
